## Create Function Cleans Building Unit/Data Clean

In [ ]:
county_list = 

In [ ]:
## select only multi-family data from Zillow
zillow_multi = zillow[zillow['type'] == "Multi"]
zillow_multi = zillow_multi[zillow_multi['code'] != "RR106"]

## crop only to residential parcels
# keep the indices where multi-family homes match to parcels
valid_parcels = parcels.sjoin(zillow_multi, predicate="contains").index.unique()

# select the parcels that match these indices
parcels_res = parcels.loc[valid_parcels]

# confirm that joining with Zillow decreases the number of parcels
assert len(parcels_res) < len(parcels)

# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(parcels_res, predicate="intersects").index.unique()
buildings_res = building.loc[valid_buildings]

## join parcels to buildings (keeping observations as parcels, but with building attributes)
# sum number of units per parcel
units_sum = parcels_res.sjoin(zillow_multi, predicate="intersects").groupby(level=0)["unit"].sum()

# join on parcels with summed number of units
parcels_res = parcels_res.join(units_sum)

# confirm that joining with Zillow decreased the number of buildings
assert len(buildings_res) < len(building)

# keep all residential buildings, and add zillow points only where they match up
building_zillow = gpd.sjoin(
    buildings_res,
    zillow_multi,
    how = "left",
    predicate = "intersects")

# reproject data frame to crs with meters as units
building_m = building_zillow.to_crs("EPSG:6933")

# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

# 6. keep only observations with unit data
building_w_units = building_m[~building_m['unit'].isna()]

assert building_w_units['unit'].isna().sum() == 0

# run model
results = smf.ols('unit ~ volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

# rerun linear regression
results_clean = smf.ols('unit ~ volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params[0]
slope = results_clean.params[1]

# extract just the multi-family homes where unit info is missing
missing_units = building_m[building_m['unit'].isna()]

# combine dataframes with missing unit data as well as outliers (since both will be predicted)
missing_outlier_units = pd.concat([building_outliers, missing_units])

assert len(missing_units) < len(missing_outlier_units)

# replace unit column with prediction
missing_outlier_units_pred = missing_outlier_units.copy().drop('unit', axis = 1)

missing_outlier_units_pred = missing_outlier_units_pred.reset_index(drop=True)

missing_outlier_units_pred['unit'] = round(intercept + missing_outlier_units_pred['volume_m3'] * slope)

# combine multi-family homes data frames
multi_complete = pd.concat([building_w_units, missing_outlier_units_pred]).to_crs(zillow.crs)

# drop residual column
multi_complete = multi_complete.drop(['residual'], axis = 1)

multi_complete = multi_complete.drop(['index_right'], axis = 1)

multi_by_parcel = parcels_res.sjoin(multi_complete, predicate="intersects")

# join back to parcels
units_multi = parcels_res.sjoin(multi_complete, predicate="intersects").groupby(level=0)['unit_right'].sum()

# join on parcels with summed number of units
multi_summed_units = parcels_res.join(units_multi)

assert len(multi_summed_units) < len(multi_by_parcel)

# save all non-multi observations
non_multi = building_m[building_m['type'] != "Multi"].to_crs(zillow.crs)

# keep only variables of interest
non_multi = non_multi[['source', 'id', 'height_m', 'var', 'region', 'bbox', 'geometry', 'area_m2', 'volume_m3']]

# join Zillow data to non-multi family homes (takes ~1 minute)
non_multi_points = gpd.sjoin(
    zillow, # left df's geometry is always kept
    non_multi,
    how = "inner",
    predicate = "intersects")